# 03 · YOLO11n traffic-sign detector (stage 1 of the cascade)

Transfer learning from COCO-pretrained `yolo11n.pt`, 40 epochs, 640 px, single class `traffic_sign`.
Training command (run in the terminal):
```
yolo detect train model=yolo11n.pt data=data/processed/signs_v0/data.yaml imgsz=640 epochs=40 batch=16 seed=42 project=runs/detect name=sprint1_signs
```

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import matplotlib.pyplot as plt

# Works whether the notebook is opened from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
print("project root:", ROOT)
from PIL import Image
from IPython.display import Image as IPImage, display

RUN = ROOT / "runs/detect/runs/detect/sprint1_signs"   # Ultralytics 8.4 nests project/ inside runs/detect
print("run folder exists:", RUN.exists())

## 1 · Training curves

In [ ]:
display(IPImage(filename=str(RUN / "results.png"), width=1000))

## 2 · Evaluation on the held-out TEST split (159 photos of unseen physical signs)

In [ ]:
from ultralytics import YOLO

model = YOLO(str(RUN / "weights/best.pt"))
m = model.val(data=str(ROOT / "data/processed/signs_v0/data.yaml"), split="test", plots=False, verbose=False)
print(f"TEST  precision={m.box.mp:.3f}  recall={m.box.mr:.3f}  mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}")

## 3 · Predictions on test images

In [ ]:
test_imgs = sorted((ROOT / "data/processed/signs_v0/images/test").iterdir())[:4]
results = model.predict([str(p) for p in test_imgs], conf=0.25, verbose=False)
fig, axes = plt.subplots(1, len(results), figsize=(18, 4))
for ax, r in zip(axes, results):
    ax.imshow(r.plot()[:, :, ::-1]); ax.axis("off")   # plot() returns BGR
plt.tight_layout(); plt.show()
print(f"speed per image: {results[0].speed}")

## 4 · Cascade preview: detector box → crop → baseline CNN (Sprint 2 integration)
Note: the classifier is trained on GTSRB (German) signs and the detector images are from another country, so class predictions here only demonstrate the data flow, not final accuracy.

In [ ]:
import io
from saferoute.vision.classifier_engine import ClassifierEngine
from saferoute.vision.datasets import class_names_from_yaml

eng = ClassifierEngine(ROOT / "models/baseline_cnn.onnx", class_names_from_yaml(ROOT / "configs/classes.yaml"))
for img_path, r in zip(test_imgs, results):
    im = Image.open(img_path).convert("RGB")
    for box, conf in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist()):
        buf = io.BytesIO()
        im.crop(tuple(int(v) for v in box)).save(buf, format="PNG")
        out = eng.predict(buf.getvalue())
        print(f"{img_path.name[:28]:28s} detector conf={conf:.2f} -> classifier {out['label']} ({out['confidence']:.2f})")